In [1]:
!pip install textstat


^C


In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import textstat

nltk.download('vader_lexicon')

# Load Dataset
df = pd.read_csv(r"C:\Users\janan\OneDrive\Documents\FakeNewsProject\WELFakeDataset.csv")
df.dropna(subset=['text', 'label'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Optional: Uncomment below if you want to limit to 25000 rows for speed
df = df.sample(5000, random_state=42)

# Feature Functions
def get_sentiment(texts):
    sid = SentimentIntensityAnalyzer()
    return [sid.polarity_scores(str(text))['compound'] for text in texts]

def get_readability(texts):
    return [textstat.flesch_reading_ease(str(text)) for text in texts]

def get_word_count(texts):
    return [len(str(text).split()) for text in texts]

def get_avg_word_length(texts):
    return [np.mean([len(word) for word in str(text).split()]) if text.split() else 0 for text in texts]

# Extract Features
df['sentiment'] = get_sentiment(df['text'])
df['readability'] = get_readability(df['text'])
df['word_count'] = get_word_count(df['text'])
df['avg_word_len'] = get_avg_word_length(df['text'])

# Separate Inputs and Labels
X_text = df['text']
X_extra = df[['sentiment', 'readability', 'word_count', 'avg_word_len']]
y = df['label']

# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
X_tfidf = tfidf_vectorizer.fit_transform(X_text)

# Combine TF-IDF with extra features
from scipy.sparse import hstack
X_combined = hstack([X_tfidf, np.array(X_extra)])

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# Train Random Forest Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f"✅ Accuracy: {acc:.4f}")

# Save Model and Vectorizer
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

print("✅ Model and vectorizer saved successfully!")


In [3]:
# Download sentiment analyzer data
nltk.download('vader_lexicon')
# Load Dataset
df = pd.read_csv(r"C:\Users\janan\OneDrive\Documents\FakeNewsProject\WELFakeDataset.csv")
df.dropna(subset=['text', 'label'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Optional: Uncomment below if you want to limit to 25000 rows for speed
df = df.sample(5000, random_state=42)

# Feature Functions
def get_sentiment(texts):
    sid = SentimentIntensityAnalyzer()
    return [sid.polarity_scores(str(text))['compound'] for text in texts]

def get_readability(texts):
    return [textstat.flesch_reading_ease(str(text)) for text in texts]

def get_word_count(texts):
    return [len(str(text).split()) for text in texts]

def get_avg_word_length(texts):
    return [np.mean([len(word) for word in str(text).split()]) if text.split() else 0 for text in texts]

# Extract Features
df['sentiment'] = get_sentiment(df['text'])
df['readability'] = get_readability(df['text'])
df['word_count'] = get_word_count(df['text'])
df['avg_word_len'] = get_avg_word_length(df['text'])

# Separate Inputs and Labels
X_text = df['text']
X_extra = df[['sentiment', 'readability', 'word_count', 'avg_word_len']]
y = df['label']

# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
X_tfidf = tfidf_vectorizer.fit_transform(X_text)

# Combine TF-IDF with extra features
from scipy.sparse import hstack
X_combined = hstack([X_tfidf, np.array(X_extra)])

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# Train Random Forest Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f"✅ Accuracy: {acc:.4f}")

# Save Model and Vectorizer
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

print("✅ Model and vectorizer saved successfully!")


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\janan\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


✅ Accuracy: 0.8720
✅ Model and vectorizer saved successfully!
